# 02 — Qualidade e chave lógica

**Objetivo:** testar duplicatas, candidatos a chave primária e regras antes da integração histórica no atlas.

**Hipótese de chave:** `Data` + `CodInstalacao` + `Tag` + `GrupoDeProdutos`

In [ ]:
from pathlib import Path

import pandas as pd

RAW_DIR = Path("../../..").resolve() / "data" / "raw" / "tancagem-abastecimento"
df = pd.read_csv(RAW_DIR / "2026" / "janeiro.csv", encoding="utf-8")
KEY = ["Data", "CodInstalacao", "Tag", "GrupoDeProdutos"]

In [ ]:
n = len(df)
n_unique = df[KEY].drop_duplicates().shape[0]
dup = df[df.duplicated(KEY, keep=False)].sort_values(KEY)

print(f"Linhas: {n:,}")
print(f"Chaves únicas ({'+'.join(KEY)}): {n_unique:,}")
print(f"Duplicatas na chave: {n - n_unique:,}")
dup.head(10)

In [ ]:
print("Nulos por coluna:")
print(df.isna().sum())

cnpj_len = df["Cnpj"].astype(str).str.len()
print("\nCNPJ — comprimento (moda):", cnpj_len.mode().iloc[0])
print(cnpj_len.value_counts().head())

print("\nTancagemM3 <= 0:", (df["TancagemM3"] <= 0).sum())

In [ ]:
# Agregação correta: instalação no snapshot
by_inst = df.groupby(["Data", "CodInstalacao"], as_index=False)["TancagemM3"].sum()
print("Instalações no snapshot:", len(by_inst))
print("Top 5 instalações por m³:")
by_inst.nlargest(5, "TancagemM3")